# AETHER — Stage 4 aether_api Real-Model Smoke Test

Первый прогон продуктового API-слоя (`aether_api`) на настоящем Qwen — не на `ScriptedSharedBackend`. Поднимает реальный `uvicorn`-сервер поверх `InterleavedDecodeScheduler` и стримит `POST /v1/turns` по-настоящему (через TCP-сокет, не in-process TestClient — тот буферизует весь ответ и не показывает живую доставку событий).

Клонирует ветку `product/api` (не `main` — там только research-ядро).

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # @param {type:"string"}
BRANCH = "product/api"  # @param {type:"string"}
MODEL_ID = "Qwen/Qwen3-1.7B"  # @param {type:"string"}
TOOL_LATENCY_MS = 1500  # @param {type:"integer"}
# float16: T4 (Turing) has no native bf16 tensor cores, and Qwen3 checkpoints
# often default to bf16 - "auto" risks slow/unstable compute on T4. Safe on A100 too.
DTYPE = "float16"  # @param {type:"string"}

if "YOUR_USERNAME" in REPO_URL:
    raise ValueError("Укажи настоящий REPO_URL")

In [ ]:
import os, subprocess, sys
from pathlib import Path

subprocess.run(["nvidia-smi"], check=False)
repo_dir = Path("/content/aether")
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{repo_dir}[dev,ml,api]"], check=True)
print("Commit:")
subprocess.run(["git", "rev-parse", "HEAD"], check=True)
print("Branch:")
subprocess.run(["git", "branch", "--show-current"], check=True)

In [ ]:
artifacts = repo_dir / "artifacts" / "colab-stage4"
artifacts.mkdir(parents=True, exist_ok=True)
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
(artifacts / "tests.log").write_text(tests.stdout, encoding="utf-8")
print(tests.stdout)
if tests.returncode != 0:
    raise RuntimeError("Tests failed")

In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(repo_dir / "src")
command = [
    sys.executable, "-m", "aether_api.experiments.colab_stage4",
    "--allow-download",
    "--model", MODEL_ID,
    "--tool-latency-ms", str(TOOL_LATENCY_MS),
    "--dtype", DTYPE,
    "--output-dir", str(artifacts),
]
run = subprocess.run(command, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(artifacts / "model_run.log").write_text(run.stdout, encoding="utf-8")
print(run.stdout)
print("Exit code:", run.returncode)

In [ ]:
import json

report_path = artifacts / "report.json"
if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print("Status:", report.get("status"))
    print("HTTP status:", report.get("http_status"))
    print("Proof:", json.dumps(report.get("proof", {}), indent=2))
    print("Error:", report.get("error", ""))
    print()
    for event in report.get("events", []):
        print(f"{event['arrived_ms']:.1f}ms  {event['type']}  {event['payload']}")

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/aether-colab-stage4-logs", "zip", root_dir=artifacts)
print(archive)
files.download(archive)